In [2]:
%%capture
!pip install -U sagemaker

In [3]:
import numpy as np
from sagemaker import get_execution_role
import sagemaker
from sagemaker.sklearn.estimator import SKLearn
from sagemaker.sklearn.model import SKLearnModel
from time import gmtime, strftime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
import datetime
import time
import tarfile
import boto3
import pandas as pd

/opt/conda/lib/python3.11/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /root/.config/sagemaker/config.yaml


## Setup

In [4]:
FRAMEWORK_VERSION = "0.23-1"
artifact = "s3://sagemaker-us-east-1-875716602731/RF-custom-sklearn-2024-12-27-09-49-12-814/output/model.tar.gz"

## Deploy

In [5]:
model_name = "Custom-sklearn-model-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
model = SKLearnModel(
    name=model_name,
    model_data=artifact,
    role=get_execution_role(),
    source_dir="code",
    entry_point="script.py",
    framework_version=FRAMEWORK_VERSION,
)

In [6]:
endpoint_name = "Custom-sklearn-model-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
print("EndpointName={}".format(endpoint_name))

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
    endpoint_name=endpoint_name,
)

EndpointName=Custom-sklearn-model-2024-12-27-15-47-59


[12/27/24 15:48:00] INFO     Creating model with name: Custom-sklearn-model-2024-12-27-15-47-58     ]8;id=431628;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=519126;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py#4094\4094]8;;\

[12/27/24 15:48:01] INFO     Creating endpoint-config with name                                     ]8;id=585584;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=200619;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py#5889\5889]8;;\
                             Custom-sklearn-model-2024-12-27-15-47-59                                              

                    INFO     Creating endpoint with name Custom-sklearn-model-2024-12-27-15-47-59   ]8;id=36640;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=170163;file:///opt/conda/lib/python3.11/site-packages/sagemaker/session.py#4711\4711]8;;\

-------!

## Test Endpoint

In [50]:
data = pd.read_csv("/root/SKLearn/processed_data/test-V-1.csv")

In [51]:
X = data[data.columns[:-1]]
y = data[data.columns[-1]]

In [39]:
# y_hat = predictor.predict(X[0:2].values.tolist())
y_hat = predictor.predict(X[0:2].values.tolist())

In [43]:
print(f"Predicted values: {y_hat}")
print(f"Predicted values: {y[0:2].values.tolist()}")

Predicted values: [3 0]
Predicted values: [3, 0]


## Endpoint invocation using ReST API

In [7]:
import requests

# API URL
url = "https://45gui6rn58.execute-api.us-east-1.amazonaws.com/stage_1/dev"

# Headers
headers = {
    "Content-Type": "text/csv"
}

# Body (CSV string)
body = "1454,1,0.5,1,1,0,34,0.7,83,4,3,250,1033,3419,7,5,5,1,1,0"

# POST request
response = requests.post(url, headers=headers, data=body)

# Check the response
if response.status_code == 200:
    print("Success:", response.text)
else:
    print("Failed:", response.status_code, response.text)


Success: {"Prediction": [3]}
